# Глава 4: Разделение данных и инструкций

- [Урок](#lesson)
- [Упражнения](#exercises)
- [Пример игровой площадки](#example-playground)

## Настройка

Запустите следующую ячейку настройки, чтобы загрузить ключ API и установить вспомогательную функцию get_completion.

In [ ]:
%pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## Урок

Зачастую нам не хочется писать полные подсказки, а вместо этого нужны **шаблоны подсказок, которые можно изменить позже, добавив дополнительные входные данные, перед отправкой Клоду**. Это может пригодиться, если вы хотите, чтобы Claude каждый раз делал одно и то же, но данные, которые Claude использует для своей задачи, могут каждый раз быть разными. 

К счастью, мы можем сделать это довольно легко, ** отделив фиксированный скелет приглашения от переменного пользовательского ввода, а затем подставив пользовательский ввод в приглашение ** перед отправкой полного приглашения Клоду. 

Ниже мы шаг за шагом рассмотрим, как написать заменяемый шаблон приглашения, а также как заменить вводимые пользователем данные.

### Примеры

В этом первом примере мы просим Клода действовать как генератор шума животных. Обратите внимание, что полное приглашение, отправленное Клоду, представляет собой просто «PROMPT_TEMPLATE», замененное входными данными (в данном случае «Корова»). Обратите внимание, что слово «Корова» заменяет заполнитель «ANIMAL» через f-строку, когда мы распечатываем полное приглашение.

**Примечание.** На практике вам не обязательно называть переменную-заполнитель каким-либо конкретным образом. В этом примере мы назвали его «ANIMAL», но с такой же легкостью мы могли бы назвать его «CREATURE» или «A» (хотя, как правило, хорошо, чтобы имена ваших переменных были конкретными и релевантными, чтобы ваш шаблон приглашения было легко понять даже без подстановки, просто для удобства анализа пользователем). Просто убедитесь, что любое имя вашей переменной соответствует тому, что вы используете для f-строки шаблона приглашения.

In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Зачем нам нужно разделять и заменять входные данные таким образом? Что ж, **шаблоны подсказок упрощают выполнение повторяющихся задач**. Допустим, вы создаете структуру подсказки, которая предлагает сторонним пользователям отправлять контент в подсказку (в данном случае животное, звук которого они хотят издавать). Этим сторонним пользователям не нужно писать или даже видеть полное приглашение. Все, что им нужно сделать, это заполнить переменные.

Мы делаем эту замену здесь, используя переменные и f-строки, но вы также можете сделать это с помощью метода format().

**Примечание.** Шаблоны подсказок могут иметь любое количество переменных!

При вводе таких переменных-заменителей очень важно **убедиться, что Клод знает, где начинаются и заканчиваются переменные** (а не инструкции или описания задач). Давайте рассмотрим пример, где нет разделения между инструкциями и подстановочной переменной.

Нашему человеческому глазу в приведенном ниже шаблоне подсказки очень ясно, где начинается и заканчивается переменная. Однако в полностью замененном приглашении это разграничение становится неясным.

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Здесь **Клод думает, что «Йоу, Клод» — это часть письма, которое он должен переписать**! Вы можете это заметить, потому что его переписывание начинается с «Дорогого Клода». Для человеческого глаза это ясно, особенно в шаблоне приглашения, где начинается и заканчивается электронное письмо, но в приглашении после замены это становится гораздо менее понятным.

Как нам это решить? **Оберните входные данные тегами XML**! Мы сделали это ниже, и, как вы можете видеть, в выводе больше нет «Дорогой Клод».

[Теги XML](https://docs.anthropic.com/claude/docs/use-xml-tags) представляют собой теги в угловых скобках, например `<tag></tag>`. Они идут парами и состоят из открывающего тега, например <tag>, и закрывающего тега, отмеченного /, например </tag>. Теги XML используются для переноса содержимого, например: <tag>content</tag>.

**Примечание.** Хотя Claude может распознавать и работать с широким спектром разделителей и разделителей, мы рекомендуем вам **использовать в качестве разделителей именно XML-теги** для Claude, поскольку Claude был специально обучен распознавать XML-теги как механизм быстрой организации. Помимо вызова функций, **не существует специальных XML-тегов, которым обучал Клода и которые вы могли бы использовать для максимального повышения производительности**. Мы намеренно сделали Клода очень гибким и настраиваемым.

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Давайте посмотрим еще один пример того, как теги XML могут нам помочь. 

В следующем приглашении **Клод неправильно интерпретирует, какая часть приглашения является инструкцией, а какая — вводом**. Он ошибочно считает `Each is about an animal, like rabbits` частью списка из-за форматирования, хотя пользователь (тот, кто заполнил переменную `SENTENCES`), предположительно, этого не хотел.

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Чтобы это исправить, нам просто нужно **окружить вводимые пользователем предложения тегами XML**. Это показывает Клоду, где начинаются и заканчиваются входные данные, несмотря на вводящий в заблуждение дефис перед Each is about an animal, like rabbits..

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

**Примечание.** В неправильной версии подсказки «Каждый из них посвящен животному» нам пришлось включить дефис, чтобы Клод ответил неправильно так, как мы хотели в этом примере. Это важный урок о подсказках: **мелкие детали имеют значение**! Всегда стоит **очистить подсказки от опечаток и грамматических ошибок**. Claude чувствителен к шаблонам (в первые годы своего существования, до тонкой настройки, это был простой инструмент прогнозирования текста), и он с большей вероятностью будет допускать ошибки, когда вы делаете ошибки, умнее, когда вы звучите умно, глупее, когда вы звучите глупо, и так далее.

Если вы хотите поэкспериментировать с подсказками к уроку, не меняя никакого содержания выше, прокрутите блокнот урока до конца, чтобы перейти к [**Пример игровой площадки**](#example-playground).

---

## Упражнения
- [Упражнение 4.1 – Тема Хайку](#exercise-41---haiku-topic)
- [Упражнение 4.2. Вопрос о собаке с опечатками](#exercise-42---dog-question-with-typos)
- [Упражнение 4.3. Вопрос о собаках, часть 2] (#exercise-42---dog-question-part-2)

### Упражнение 4.1 — Тема Хайку
Измените `PROMPT` так, чтобы это был шаблон, который будет принимать переменную с именем `TOPIC` и выводить хайку по этой теме. Это упражнение предназначено только для того, чтобы проверить ваше понимание структуры шаблонов переменных с f-строками.

In [ ]:
# Variable content
TOPIC = "Pigs"

# Prompt template with a placeholder for the variable content
PROMPT = f""

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("pigs", text.lower()) and re.search("haiku", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
from hints import exercise_4_1_hint; print(exercise_4_1_hint)

### Упражнение 4.2 — Вопрос о собаке с опечатками
Исправьте `PROMPT`, добавив теги XML, чтобы Клод дал правильный ответ. 

Постарайтесь больше ничего не менять в подсказке. Беспорядочное и полное ошибок письмо сделано намеренно, поэтому вы можете увидеть, как Клод реагирует на такие ошибки.

In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
from hints import exercise_4_2_hint; print(exercise_4_2_hint)

### Упражнение 4.3 — Вопрос о собаках, часть 2
Исправьте `PROMPT` **БЕЗ** добавления тегов XML. Вместо этого удалите из подсказки только одно или два слова.

Как и в случае с приведенными выше упражнениями, постарайтесь больше ничего не менять в подсказке. Это покажет вам, какой язык Клод может анализировать и понимать.

In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
from hints import exercise_4_3_hint; print(exercise_4_3_hint)

### Поздравляю!

Если вы выполнили все упражнения до этого момента, вы готовы перейти к следующей главе. Приятного подсказки!

---

## Пример игровой площадки

Это область, где вы можете свободно экспериментировать с примерами подсказок, представленными в этом уроке, и настраивать подсказки, чтобы увидеть, как они могут повлиять на ответы Клода.

In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))